# 02 · Sampling Experiments — the (SI-SDR, SLR) plane, the exposure→leakage mechanism, and the policy to ship

This notebook runs the pre-registered experiment: train the compact separator under each of
the four chunk-sampling policies (uniform / energy / drop / curriculum), assemble the
**headline (SI-SDR, SLR) plane** with per-seed scatter and the do-nothing / oracle-IRM
anchors, read the θ-sensitivity grid, tie realized silent-exposure to SLR (the mechanism),
run the single test pass with paired stats, and pick the policy StemCraft ships. The
training/test cells are **RUN LATER** (GPU); every analysis function is in `singnet`,
exercised on constructed/illustrative data here so the logic is visible before any GPU spend.

The notebook ships **un-executed**. Runtimes on the RUN-LATER banners are from MASTER_PLAN §7.

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §3 (pre-registration, recapped
  verbatim below), §4.2 (run matrix), §6 (protocol), §7 (budget), §13 (interpretation);
  [`../THEORY.md`](../THEORY.md) §4 (exposure distributions), §6 (statistics + selection guard).
- **Data prep is *not* repeated here** — reused from Direction 01
  ([`../../01-loss-function-study/notebooks/01_data_and_eda.ipynb`](../../01-loss-function-study/notebooks/01_data_and_eda.ipynb));
  this direction adds only the energy-profile pass (`scripts/prepare_data.py --write-energy-profiles`).
- **Code, not prose, is authoritative:** the metric is `singnet.metrics.slr`; the policies are
  `singnet.data.sampling`; σ_seed is `singnet.analysis.pooled_seed_sigma`; runs are
  `scripts/run_sweep.py --direction 08`.

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU.
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"] = "/content/drive/MyDrive/musdb_shards"
print("Bootstrap cell — run on Colab only (see comments). No-op here.")

## 1 · Pre-registration recap (MASTER_PLAN §3, verbatim)

> Notation: $q(\text{policy})$ = mean best-checkpoint validation vocals SI-SDR;
> $\ell(\text{policy})$ = mean validation SLR (θ = −60, valid tracks). Two pooled seed
> noises from the three 3-seed cells (uniform, energy, drop):
> $\sigma_{\text{seed}}^{q}$ for SI-SDR and $\sigma_{\text{seed}}^{\ell}$ for SLR.
>
> **H-08a (quality).** Energy-weighted sampling beats uniform on overall quality:
> $q(\text{energy}) - q(\text{uniform}) > \sigma_{\text{seed}}^{q}$.
> Refuted iff ≤; each direction pre-written (§13).
>
> **H-08b (the cost of dropping silence).** Discarding silent chunks poisons silence
> behavior: $\ell(\text{drop}) - \ell(\text{uniform}) > \sigma_{\text{seed}}^{\ell}$
> (drop leaks detectably more). Refuted iff ≤ — i.e. the model outputs silence "for
> free" without ever training on it (a genuinely interesting negative about mask
> models; pre-written).
>
> **The headline exhibit (pre-registered format):** the **(SI-SDR, SLR) plane** — four
> policy points with per-seed scatter, plus the do-nothing (0 dB SLR) and oracle-IRM
> anchors. **We pre-commit to NOT scalarizing** the two axes into one score; the
> tradeoff, if it exists, is the finding. Curriculum's pre-registered role: does an
> anneal capture energy's quality gain *without* drop's leakage cost (dominates both on
> the plane / sits between / dominates neither — three pre-written readings).
>
> **Descriptive secondaries:** θ-sensitivity of every conclusion ({−50, −60, −70});
> per-policy exposure statistics (fraction of silent chunks actually seen during
> training — logged, closing the loop on the THEORY §4 exposure predictions); SLR of
> Direction 01's arms at ε... (cross-feed: the `sisdr`-loss guard's skip rate vs SLR,
> descriptive); museval SDR secondary table (expected ≈ blind to the policy differences
> on silence — itself worth one line if confirmed).

## 2 · Run matrix + shared-cell accounting (MASTER_PLAN §4.2)

Eight new GPU runs; the **uniform** arm is the shared Direction-01 `l1mag` cell (reused, not
retrained — its config hash equals D01's, asserted below). The cell prints the matrix and the
policy/θ/λ each config resolves to — the same information `run_sweep.py --direction 08
--dry-run` shows, but inline.

In [ ]:
# CPU-runnable now: the run matrix, the shared-cell hash equality, and each config's policy.
from singnet.utils.config import resolve_config, hash_config, sampling_policy

D08 = "08-silence-leakage/configs"
D01 = "01-loss-function-study/configs/l1mag_seed0_reduced.yaml"

shared = hash_config(resolve_config(D01))
base = hash_config(resolve_config(f"{D08}/base.yaml"))
print(f"uniform arm == D01 l1mag shared cell:  {base == shared}  (0 new runs for uniform)\n")

matrix = ["energy_seed0", "energy_seed1", "energy_seed2",
          "drop_seed0", "drop_seed1", "drop_seed2",
          "curriculum_seed0", "curriculum_seed1"]
print(f"{'config':<20}{'policy':<12}{'loss':<7}{'seed':<5}{'hash'}")
for name in matrix:
    cfg = resolve_config(f"{D08}/{name}.yaml")
    print(f"{name:<20}{sampling_policy(cfg)['policy']:<12}{cfg.get('loss'):<7}{cfg['seed']:<5}{hash_config(cfg)}")
print("\n8 new runs (energy x3, drop x3, curriculum x2) + 2 budget-gated sisdr contingencies.")

In [ ]:
# ⚠️ RUN THIS LATER (GPU) — the energy-profile prep + the 8 runs.
# Prereqs: D01 data prep done (shards on disk). Profiles are CPU-minutes; each run ~1.3-1.8 h
# at REDUCED (16k steps); 8 runs ~10-14 T4-hours total (MASTER_PLAN §7).
#
#   # 0. energy profiles (CPU, once):
#   !python scripts/prepare_data.py --write-energy-profiles --out $SHARD_ROOT
#   # 1. dry-run (CPU): policy/theta/lambda + curriculum lambda(t) endpoints per config:
#   !python scripts/run_sweep.py --direction 08 --dry-run
#   # 2. the 8 runs (GPU); fully resumable (a Colab disconnect costs minutes):
#   !python scripts/run_sweep.py --direction 08 --stage reduced
print("Profiles + 8 runs — RUN LATER (GPU). ~10-14 T4-h (MASTER_PLAN §7).")

## 3 · Headline figure — the (SI-SDR, SLR) plane (MASTER_PLAN §3)

The pre-registered headline: each policy is a point (mean over seeds) with **per-seed
scatter**, on axes (vocals SI-SDR →, SLR ↓). The **do-nothing** anchor sits at **SLR = 0 dB**
exactly (THEORY §3.2); the **oracle-IRM** anchor is the mask-family lower-leakage bound.
**The two axes are NOT scalarized** — the tradeoff geometry is the finding. The cell below
draws the plane with a **schematic** layout (clearly labeled) so the plotting logic is
complete un-run; **RUN LATER** it reads the real per-seed points from the val CSVs.

In [ ]:
# ⚠️ RUN THIS LATER for real points — the headline (SI-SDR, SLR) plane.
# The layout below is SCHEMATIC (pre-registered geometry; NOT results) so the figure code is
# visible un-run. RUN LATER: replace `schematic` with per-seed (sisdr, slr@-60) from the val
# per-track CSVs (evaluate.py --slr), keeping the do-nothing (0 dB) and oracle-IRM anchors.
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(8)
# schematic centroids illustrating the H-08a (energy right of uniform) and H-08b (drop above
# uniform) predictions; per-seed scatter is jitter around each centroid. NOT measured values.
centroids = {  # (sisdr_center, slr_center, n_seeds, color)
    "uniform":    (5.0, -12.0, 3, "#8899aa"),
    "energy":     (5.6, -11.8, 3, "#2b6cb0"),
    "drop":       (5.5,  -8.5, 3, "#c0392b"),
    "curriculum": (5.6, -12.0, 2, "#e0b030"),
}
fig, ax = plt.subplots(figsize=(6.6, 5.2))
for name, (qc, lc, n, c) in centroids.items():
    q = qc + 0.12 * rng.standard_normal(n)
    l = lc + 0.35 * rng.standard_normal(n)
    ax.scatter(q, l, color=c, s=28, alpha=0.5)                       # per-seed scatter
    ax.scatter(qc, lc, color=c, s=150, edgecolor="k", zorder=5, label=name)  # seed mean
ax.axhline(0.0, color="k", ls="--", lw=1)                            # do-nothing: SLR = 0 dB
ax.text(ax.get_xlim()[0] + 0.05, 0.3, "do-nothing anchor (SLR = 0 dB)", fontsize=8)
ax.scatter([8.5], [-45.0], marker="*", s=240, color="gold", edgecolor="k", label="oracle IRM")
ax.set_xlabel("vocals SI-SDR (dB)  —  quality, higher better")
ax.set_ylabel("SLR @ theta = -60 dB  —  leakage, lower better")
ax.set_title("(SI-SDR, SLR) plane  [SCHEMATIC — real points fill in after the runs]")
ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=0.3); fig.tight_layout(); plt.show()
print("Headline plane — RUN LATER for real points. The tradeoff is NOT scalarized (MASTER_PLAN §3).")

## 4 · θ-sensitivity grid (MASTER_PLAN §3 secondary; THEORY §3.6)

Every conclusion is reported at **θ ∈ {−50, −60, −70}**: the same plane (or the (drop −
uniform) SLR gap) is redrawn at each threshold, so no result rests on a single silence
definition. **RUN LATER** — reads the val CSVs, which already carry `slr_m50/m60/m70`.

In [ ]:
# ⚠️ RUN THIS LATER — the theta-sensitivity grid (evaluate.py --slr writes all three thetas).
# For theta in (-50, -60, -70): recompute each policy's mean SLR and redraw the (drop - uniform)
# gap; H-08b's verdict must hold (or its sign be reported) across all three (MASTER_PLAN §3).
#
#   import pandas as pd
#   df = pd.read_csv("08-silence-leakage/results/valid_per_track.csv")   # has slr_m50/m60/m70
#   for col in ("slr_m50", "slr_m60", "slr_m70"):
#       means = df.groupby("system")[col].mean()   # valid-n excluded via NaN automatically
#       ...  # bar/point grid of policy SLR at each theta; annotate valid-n
print("theta-sensitivity grid — RUN LATER (val CSVs carry slr_m50/m60/m70).")

## 5 · Exposure → leakage: the mechanism (THEORY §4; MASTER_PLAN §6)

The loop logs the **realized** silent-chunk exposure every 500 steps (`sampling_exposure.csv`)
and the run mean in the registry (`silent_exposure_observed`). THEORY §4 predicts uniform ≈ φ,
drop = 0, energy ≈ λφ, curriculum front-loaded. This cell ties **realized exposure → measured
SLR** across arms — the mechanism link, and the §13 diagnostic of whether the policies even
differed in practice. **RUN LATER** (reads the telemetry + registry).

In [ ]:
# ⚠️ RUN THIS LATER — exposure telemetry vs SLR (the mechanism link, THEORY §4).
# Confirms the closed-form exposure predictions against the run, then correlates realized
# exposure with SLR: lower exposure (drop) should track higher (worse) SLR if H-08b holds.
#
#   import pandas as pd
#   reg = pd.read_csv("08-silence-leakage/results/registry.csv")
#   ax = reg.plot.scatter(x="silent_exposure_observed", y="best_val_slr")  # per run
#   # overlay the THEORY predictions: uniform=phi (base rate), drop=0, energy~0.1*phi,
#   # curriculum~0.325*phi (the lambda-bar integral, THEORY §5). Annotate each policy.
#   # also load a per-run sampling_exposure.csv to show curriculum's front-loaded anneal.
print("Exposure -> SLR mechanism — RUN LATER (reads sampling_exposure.csv + registry).")

## 6 · Test pass + paired statistics (MASTER_PLAN §6)

Exactly one test pass: the best checkpoints (9 seed cells + 2 curriculum) + do-nothing +
oracle IRM on the 50 test tracks, scoring vocals/accomp SI-SDR, **SLR at all three θ**, and
the museval SDR secondary. Confirmatory inference is **paired bootstrap + Wilcoxon** on the
two pre-registered pairs: (energy − uniform) on SI-SDR, (drop − uniform) on SLR. σ_seed comes
from `singnet.analysis.pooled_seed_sigma`. **RUN LATER** (GPU minutes for the pass).

In [ ]:
# ⚠️ RUN THIS LATER (GPU minutes) — the single test pass + paired stats.
#   # 1. score the 11 checkpoints + anchors on the 50 test tracks, with SLR at all three theta:
#   !python scripts/evaluate.py --direction 08 --split test --slr \
#       --shard-root $SHARD_ROOT --splits-csv 01-loss-function-study/configs/splits.csv \
#       --oracles --output-dir 08-silence-leakage/results
#   # 2. pooled seed sigma from the three 3-seed cells (SI-SDR and SLR):
#   #    from singnet.analysis import pooled_seed_sigma
#   #    sigma_q = pooled_seed_sigma(reg, metric="best_val_sisdr", by="policy", cells=["uniform","energy","drop"])
#   #    sigma_l = pooled_seed_sigma(reg, metric="best_val_slr",   by="policy", cells=["uniform","energy","drop"])
#   # 3. paired bootstrap + Wilcoxon on the two pre-registered pairs (numpy inline + scipy):
#   #    delta_q = test_sisdr["energy"] - test_sisdr["uniform"]   # H-08a
#   #    delta_l = test_slr["drop"]     - test_slr["uniform"]     # H-08b
#   #    # from scipy.stats import wilcoxon; wilcoxon(delta_q); wilcoxon(delta_l)
print("Test pass + paired stats — RUN LATER (GPU minutes; MASTER_PLAN §6).")

## 7 · Interpretation branches (MASTER_PLAN §13 — labeled stubs, filled at analysis freeze)

The verdict is pre-committed to the outcome, so no post-hoc story is possible. Each branch
below is a **stub** to be completed with the frozen numbers; the full prose lives in
`paper/PAPER.md`.

### 7.1 H-08a **supported** + H-08b **supported** — the predicted tradeoff
Sampling is a real lever with a real price; the plane shows the frontier. **Consequence:**
ship `energy` (quality) unless karaoke-critical, then `uniform`/`curriculum`; SLR enters the
project's standard eval battery. _[fill: Δq, Δℓ, plane]_

### 7.2 H-08a **supported** + H-08b **refuted** — the free lunch
Focus sampling on vocals; silence behavior survives anyway (the mask prior suffices).
**Consequence:** ship `energy` everywhere; the "never seen silence" fear is retired for mask
models at this scale. _[fill: Δq, the null Δℓ vs σ]_

### 7.3 H-08a **refuted** + H-08b **supported** — pure downside risk
Sampling doesn't buy quality, but dropping still poisons silence. **Consequence:** ship
`uniform`; publish the caution against activity filtering. _[fill: null Δq, Δℓ]_

### 7.4 **Both refuted** — chunk sampling is a non-lever at this scale
16 k steps may see enough of everything. **Consequence:** honest null; the exposure telemetry
says whether the policies even differed in practice (the diagnostic). _[fill: both nulls +
realized exposures]_

### 7.5 Curriculum **dominates** / **is dominated** on the plane
Dominates: annealed exposure captures both goods → ship `curriculum`, flag schedule
sensitivity as future work. Dominated: added complexity, no win → one-line negative. _[fill:
curriculum's plane position vs energy/uniform]_

### 7.6 SLR **invalid on too many tracks** (G2 fail path)
MUSDB's silences are rarer/shorter than assumed. **Consequence:** θ/L_min sensitivity becomes
primary; the metric definition is revised **only** via `results/DEVIATIONS.md` with reasons.
_[fill: valid-n at each θ]_

## 8 · Conclusions + the StemCraft recommendation

_Filled at analysis freeze from the frozen plane and the two paired tests._

- **Which policy ships in SingNet training?** The plane + §7 branch decide: `energy` if it is
  a free lunch (7.2) or the quality win outweighs the karaoke cost (7.1, non-karaoke);
  `uniform`/`curriculum` when ghost-vocals are product-critical (7.1 karaoke, 7.5).
- **Does SLR join the standard eval battery?** If the tradeoff is real (7.1), yes — SLR is the
  axis SDR cannot see, and every future direction reports it alongside SI-SDR.
- **The honest caveat:** SLR is an energy claim (timbre-blind, phase-invisible, THEORY §3.8);
  the umbrella listening check carries the perceptual verdict on the shipped policy.